In [3]:
# ============================================================
# MRB1-DIAG — Continuous Pedagogical Score Audit
# ============================================================
#
# PURPOSE
# -------
# MRB1's hard 3-class predictions rarely/never choose the middle
# label ("To some extent") for some tasks. But LinTS will NOT use
# hard labels. It will use MRB1's continuous expected score:
#
#   score = 0.0 * P(No)
#         + 0.5 * P(To some extent)
#         + 1.0 * P(Yes)
#
# Therefore this diagnostic asks the more relevant question:
#
#   Do the MRB1 probabilities produce a meaningful ordered
#   continuous pedagogical-quality signal?
#
# This notebook does NOT train, tune, or touch MRBench V3 test.
# It only audits the already-frozen MRB1 epoch-4 validation
# predictions.
#
# Required Kaggle input:
# - Add the SAVED OUTPUT of the completed MRB1 notebook.
# - It must contain:
#     mrb1_results/mrb1_epoch4_validation_predictions.jsonl
#
# Metrics:
# - MAE / RMSE against ordinal targets {0, 0.5, 1}
# - Spearman rank correlation
# - Pearson correlation
# - Pairwise ordinal concordance
# - Regression-style calibration error
# - Class-conditional score means/medians
# - Adjacent-class AUCs:
#       No vs Partial
#       Partial vs Yes
#       No vs Yes
#
# IMPORTANT
# ---------
# This is a diagnostic, NOT a new MRB model experiment.
# ============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import spearmanr, pearsonr
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    roc_auc_score,
)

TASKS = [
    "Mistake_Identification",
    "Mistake_Location",
    "Providing_Guidance",
    "Actionability",
]

LABEL_VALUE = {
    "No": 0.0,
    "To some extent": 0.5,
    "Yes": 1.0,
}

RESULT_DIR = Path(
    "/kaggle/working/mrb1_continuous_score_audit"
)
RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("=" * 78)
print("MRB1-DIAG — CONTINUOUS SCORE AUDIT")
print("=" * 78)
print("Training performed: False")
print("MRBench V3 test touched: False")


MRB1-DIAG — CONTINUOUS SCORE AUDIT
Training performed: False
MRBench V3 test touched: False


In [4]:
# ============================================================
# LOCATE SAVED MRB1 EPOCH-4 PREDICTIONS
# ============================================================

prediction_candidates = list(
    Path("/kaggle/input").glob(
        "**/mrb1_epoch4_validation_predictions.jsonl"
    )
)

print("Prediction candidates:")
for p in prediction_candidates:
    print(" ", p)

assert len(prediction_candidates) == 1, (
    "Expected exactly one MRB1 epoch-4 validation prediction file. "
    f"Found {len(prediction_candidates)}. "
    "Add the completed MRB1 notebook output as a Kaggle input."
)

PRED_PATH = prediction_candidates[0]

rows = []

with PRED_PATH.open(
    "r",
    encoding="utf-8",
) as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(
                json.loads(line)
            )

print("\nRows loaded:", len(rows))

assert len(rows) == 495, (
    f"Expected 495 MRB1 validation responses, got {len(rows)}."
)

required = {
    "conversation_id",
    "tutor_name",
    "true_labels",
    "predicted_labels",
    "continuous_scores",
    "probabilities",
}

assert required.issubset(rows[0].keys())

for row in rows:
    for task in TASKS:
        assert task in row["true_labels"]
        assert task in row["continuous_scores"]
        assert task in row["probabilities"]

        probs = row["probabilities"][task]

        assert set(probs.keys()) == {
            "No",
            "To some extent",
            "Yes",
        }

        prob_sum = sum(probs.values())
        assert abs(prob_sum - 1.0) < 1e-4

        recomputed = (
            0.5 * probs["To some extent"]
            + probs["Yes"]
        )

        assert abs(
            recomputed
            - row["continuous_scores"][task]
        ) < 1e-5

print("Prediction schema + probability audit: PASSED")


Prediction candidates:
  /kaggle/input/datasets/malindus/mrb1-results/mrb1_epoch4_validation_predictions.jsonl

Rows loaded: 495
Prediction schema + probability audit: PASSED


In [5]:
# ============================================================
# HELPER METRICS
# ============================================================

def safe_corr(fn, y_true, scores):
    value = fn(
        y_true,
        scores,
    )[0]

    if np.isnan(value):
        return None

    return float(value)


def pairwise_ordinal_concordance(
    y_true,
    scores,
):
    """
    Fraction of differently-labelled pairs whose score ordering
    agrees with the true ordinal ordering.

    Tied predicted scores receive 0.5 credit.
    Same-label pairs are excluded.
    """
    y_true = np.asarray(
        y_true,
        dtype=float,
    )
    scores = np.asarray(
        scores,
        dtype=float,
    )

    correct = 0.0
    total = 0

    n = len(y_true)

    for i in range(n):
        for j in range(i + 1, n):
            if y_true[i] == y_true[j]:
                continue

            true_sign = np.sign(
                y_true[i] - y_true[j]
            )

            score_sign = np.sign(
                scores[i] - scores[j]
            )

            total += 1

            if score_sign == true_sign:
                correct += 1.0
            elif score_sign == 0:
                correct += 0.5

    return (
        float(correct / total)
        if total > 0
        else None
    )


def regression_calibration_error(
    y_true,
    scores,
    n_bins=10,
):
    """
    Bin the continuous critic score and compare mean score with
    mean ordinal target in each populated bin.
    """
    y_true = np.asarray(
        y_true,
        dtype=float,
    )
    scores = np.asarray(
        scores,
        dtype=float,
    )

    edges = np.linspace(
        0.0,
        1.0,
        n_bins + 1,
    )

    error = 0.0
    rows = []

    for i in range(n_bins):
        lo = edges[i]
        hi = edges[i + 1]

        if i == n_bins - 1:
            mask = (
                (scores >= lo)
                & (scores <= hi)
            )
        else:
            mask = (
                (scores >= lo)
                & (scores < hi)
            )

        count = int(mask.sum())

        if count == 0:
            continue

        mean_score = float(
            scores[mask].mean()
        )
        mean_target = float(
            y_true[mask].mean()
        )

        bin_error = abs(
            mean_score
            - mean_target
        )

        error += (
            count / len(scores)
        ) * bin_error

        rows.append({
            "bin_low": lo,
            "bin_high": hi,
            "count": count,
            "mean_score": mean_score,
            "mean_target": mean_target,
            "abs_error": bin_error,
        })

    return float(error), rows


def adjacent_auc(
    true_labels,
    scores,
    lower_label,
    upper_label,
):
    mask = np.asarray([
        label in {
            lower_label,
            upper_label,
        }
        for label in true_labels
    ])

    labels_sub = np.asarray(
        true_labels,
        dtype=object,
    )[mask]

    scores_sub = np.asarray(
        scores,
        dtype=float,
    )[mask]

    y_binary = (
        labels_sub
        == upper_label
    ).astype(int)

    if len(
        np.unique(y_binary)
    ) < 2:
        return None

    return float(
        roc_auc_score(
            y_binary,
            scores_sub,
        )
    )


In [6]:
# ============================================================
# TASK-BY-TASK CONTINUOUS SCORE AUDIT
# ============================================================

audit = {
    "experiment": "MRB1_DIAG_ContinuousScore_Audit",
    "training_performed": False,
    "official_v3_test_used": False,
    "validation_rows": len(rows),
    "tasks": {},
}

summary_rows = []

for task in TASKS:
    true_labels = [
        row["true_labels"][task]
        for row in rows
    ]

    y_true = np.asarray(
        [
            LABEL_VALUE[label]
            for label in true_labels
        ],
        dtype=float,
    )

    scores = np.asarray(
        [
            row["continuous_scores"][task]
            for row in rows
        ],
        dtype=float,
    )

    mae = float(
        mean_absolute_error(
            y_true,
            scores,
        )
    )

    rmse = float(
        mean_squared_error(
            y_true,
            scores,
        ) ** 0.5
    )

    spearman = safe_corr(
        spearmanr,
        y_true,
        scores,
    )

    pearson = safe_corr(
        pearsonr,
        y_true,
        scores,
    )

    concordance = (
        pairwise_ordinal_concordance(
            y_true,
            scores,
        )
    )

    calibration_error, calibration_bins = (
        regression_calibration_error(
            y_true,
            scores,
            n_bins=10,
        )
    )

    class_stats = {}

    for label in [
        "No",
        "To some extent",
        "Yes",
    ]:
        mask = np.asarray(
            [
                x == label
                for x in true_labels
            ]
        )

        class_scores = scores[mask]

        class_stats[label] = {
            "count": int(mask.sum()),
            "mean_score": float(
                class_scores.mean()
            ),
            "median_score": float(
                np.median(class_scores)
            ),
            "std_score": float(
                class_scores.std()
            ),
        }

    auc_no_partial = adjacent_auc(
        true_labels,
        scores,
        "No",
        "To some extent",
    )

    auc_partial_yes = adjacent_auc(
        true_labels,
        scores,
        "To some extent",
        "Yes",
    )

    auc_no_yes = adjacent_auc(
        true_labels,
        scores,
        "No",
        "Yes",
    )

    means = [
        class_stats["No"]["mean_score"],
        class_stats[
            "To some extent"
        ]["mean_score"],
        class_stats["Yes"]["mean_score"],
    ]

    monotonic_means = bool(
        means[0]
        < means[1]
        < means[2]
    )

    result = {
        "mae": mae,
        "rmse": rmse,
        "spearman": spearman,
        "pearson": pearson,
        "pairwise_ordinal_concordance": concordance,
        "regression_calibration_error_10bin": calibration_error,
        "class_conditional_scores": class_stats,
        "auc_no_vs_partial": auc_no_partial,
        "auc_partial_vs_yes": auc_partial_yes,
        "auc_no_vs_yes": auc_no_yes,
        "class_mean_scores_monotonic": monotonic_means,
        "calibration_bins": calibration_bins,
    }

    audit["tasks"][task] = result

    summary_rows.append({
        "Task": task,
        "MAE": mae,
        "RMSE": rmse,
        "Spearman": spearman,
        "Pearson": pearson,
        "Concordance": concordance,
        "CalError": calibration_error,
        "NoMean": means[0],
        "PartialMean": means[1],
        "YesMean": means[2],
        "No-v-Partial AUC": auc_no_partial,
        "Partial-v-Yes AUC": auc_partial_yes,
        "No-v-Yes AUC": auc_no_yes,
        "MonotonicMeans": monotonic_means,
    })

summary_df = pd.DataFrame(
    summary_rows
)

print("=" * 78)
print("MRB1 CONTINUOUS-SCORE SUMMARY")
print("=" * 78)

print(
    summary_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.4f}",
    )
)


MRB1 CONTINUOUS-SCORE SUMMARY
                  Task    MAE   RMSE  Spearman  Pearson  Concordance  CalError  NoMean  PartialMean  YesMean  No-v-Partial AUC  Partial-v-Yes AUC  No-v-Yes AUC  MonotonicMeans
Mistake_Identification 0.1054 0.2023    0.5935   0.8168       0.9122    0.0212  0.2529       0.7240   0.9393            0.8661             0.8464        0.9535            True
      Mistake_Location 0.2636 0.3654    0.5460   0.5985       0.8117    0.0614  0.4440       0.7349   0.8202            0.7051             0.7375        0.8385            True
    Providing_Guidance 0.2647 0.3311    0.4959   0.5978       0.7671    0.0574  0.3815       0.7070   0.7825            0.7944             0.6244        0.8659            True
         Actionability 0.2451 0.3304    0.6694   0.6948       0.8468    0.0390  0.3005       0.6709   0.8028            0.8207             0.6893        0.9153            True


In [7]:
# ============================================================
# AGGREGATE INTERPRETATION METRICS
# ============================================================

numeric_cols = [
    "MAE",
    "RMSE",
    "Spearman",
    "Pearson",
    "Concordance",
    "CalError",
    "No-v-Partial AUC",
    "Partial-v-Yes AUC",
    "No-v-Yes AUC",
]

aggregate = {}

for col in numeric_cols:
    values = pd.to_numeric(
        summary_df[col],
        errors="coerce",
    )

    aggregate[
        f"mean_{col}"
    ] = float(
        values.mean()
    )

monotonic_count = int(
    summary_df[
        "MonotonicMeans"
    ].sum()
)

aggregate[
    "tasks_with_monotonic_class_means"
] = monotonic_count

audit["aggregate"] = aggregate

print("\n" + "=" * 78)
print("AGGREGATE")
print("=" * 78)

print(
    "Mean MAE:",
    aggregate["mean_MAE"],
)
print(
    "Mean RMSE:",
    aggregate["mean_RMSE"],
)
print(
    "Mean Spearman:",
    aggregate["mean_Spearman"],
)
print(
    "Mean pairwise concordance:",
    aggregate["mean_Concordance"],
)
print(
    "Mean continuous calibration error:",
    aggregate["mean_CalError"],
)
print(
    "Mean No-v-Partial AUC:",
    aggregate["mean_No-v-Partial AUC"],
)
print(
    "Mean Partial-v-Yes AUC:",
    aggregate["mean_Partial-v-Yes AUC"],
)
print(
    "Mean No-v-Yes AUC:",
    aggregate["mean_No-v-Yes AUC"],
)
print(
    "Tasks with ordered class-conditional means:",
    f"{monotonic_count}/4",
)

print("\nClass-conditional mean-score ordering:")
for row in summary_rows:
    print(
        f"{row['Task']:24s}: "
        f"No={row['NoMean']:.4f} < "
        f"Partial={row['PartialMean']:.4f} < "
        f"Yes={row['YesMean']:.4f} "
        f"=> {row['MonotonicMeans']}"
    )



AGGREGATE
Mean MAE: 0.21968445967544206
Mean RMSE: 0.3072993117914188
Mean Spearman: 0.5762149001176877
Mean pairwise concordance: 0.83443744410419
Mean continuous calibration error: 0.04476785692151149
Mean No-v-Partial AUC: 0.7965735298205061
Mean Partial-v-Yes AUC: 0.724403081214646
Mean No-v-Yes AUC: 0.8932771972914584
Tasks with ordered class-conditional means: 4/4

Class-conditional mean-score ordering:
Mistake_Identification  : No=0.2529 < Partial=0.7240 < Yes=0.9393 => True
Mistake_Location        : No=0.4440 < Partial=0.7349 < Yes=0.8202 => True
Providing_Guidance      : No=0.3815 < Partial=0.7070 < Yes=0.7825 => True
Actionability           : No=0.3005 < Partial=0.6709 < Yes=0.8028 => True


In [8]:
# ============================================================
# SAVE AUDIT
# ============================================================

summary_path = (
    RESULT_DIR
    / "mrb1_continuous_score_audit.json"
)

table_path = (
    RESULT_DIR
    / "mrb1_continuous_score_summary.csv"
)

with summary_path.open(
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        audit,
        f,
        indent=2,
    )

summary_df.to_csv(
    table_path,
    index=False,
)

print("\nSaved:")
print(summary_path)
print(table_path)

print("\nMRB1-DIAG COMPLETE")
print(
    "No training occurred and the official MRBench V3 test set "
    "was not used."
)



Saved:
/kaggle/working/mrb1_continuous_score_audit/mrb1_continuous_score_audit.json
/kaggle/working/mrb1_continuous_score_audit/mrb1_continuous_score_summary.csv

MRB1-DIAG COMPLETE
No training occurred and the official MRBench V3 test set was not used.
